# 0. Library

In [1]:
import pandas as pd
from pathlib import Path
import re

import importlib
import sys
sys.path.append('../') 
from features import data_utils as du
from features import data_pipeline as dp
from features import general_func as gf
import constants_data as cd

# Reload
importlib.reload(du)
importlib.reload(dp)
importlib.reload(gf)
importlib.reload(cd)

<module 'constants_data' from '/home/smira/myproject/detection_AD_with_VR_data/src/notebooks/../constants_data.py'>

# 1. Paths and Constants

In [2]:
# Constants and paths
parent_folder = Path("../..") # go 2 folder up= "../.."
df_path = parent_folder / "data" / "produced_csv" / "1.extracted_features.csv"
 
df = pd.read_csv(df_path)

df.head()

,Paitent_id,Age,Gender,dominant_hand,Sessions_Completed_out_of_4,Help_Rating_out_of_5,MoCA_Score,Tutorial_reading_time_happened,Tutorial_total_reading_time,Tutorial_mean_reading_time,...,Memory_not_dominant_hand_y_range,Memory_not_dominant_hand_z_range,Memory_dominant_hand_trigger_press_count,Memory_not_dominant_hand_trigger_press_count,Memory_dominant_hand_trigger_pressure_mean,Memory_not_dominant_hand_trigger_pressure_mean,Memory_dominant_hand_grip_count,Memory_not_dominant_hand_grip_count,Memory_dominant_hand_grip_mean,Memory_not_dominant_hand_grip_mean
0,P001,NaN,Female,Right,4,1,28.0,1,45.03,15.01,...,0.80,0.50,35,0,0.01,0.0,1275,0,0.40,0.0
1,P002,59.0,Female,Right,4,3,NaN,1,171.99,57.33,...,0.61,1.15,63,0,0.02,0.0,222,0,0.09,0.0
2,P003,82.0,Male,Right,4,4,26.0,1,338.75,112.92,...,0.75,0.61,12,0,0.00,0.0,277,0,0.09,0.0
3,P004,75.0,Male,Left,4,3,NaN,1,114.78,38.26,...,0.59,0.92,47,1,0.03,0.0,157,0,0.10,0.0
4,P005,62.0,Male,Right,4,4,27.0,1,152.90,50.97,...,0.58,0.48,28,0,0.02,0.0,125,0,0.08,0.0


In [3]:
df_cleaning = df.copy()

# 2. Data preprocessing

## 2.1 Fill missing values

In [4]:
df_cleaning = dp.check_Nan_values(original_df= df_cleaning, missing_values_threshold=0.6)

Missing values found, number of columns with Nan values : 4
All Nan values are handled successfuly.


In [5]:
df_cleaning = dp.check_columns_type(df_without_Nan= df_cleaning)

Columns before one hot encoding : 

          Total columns :376, 
          Numerical columns : 373,  
          Categorical columns : 3, 
          Unkown columns : 0
        

 Columns after one hot encoding : 

          Total columns :375, 
          Numerical columns : 375,  
          Categorical columns : 0, 
          Unkown columns : 0
        
All Categorical columns are encoded successfuly.


In [6]:
df_cleaning = dp.check_constant_column(df_without_categorical_column=df_cleaning)

Columns before removing constant columns : 375
There are 28 constant columns, removing them...

 Columns after removing constant columns : 347
All constant columns are removed successfuly.


In [7]:
df_cleaning.head()

,Age,Help_Rating_out_of_5,MoCA_Score,Tutorial_total_reading_time,Tutorial_mean_reading_time,Tutorial_max_reading_time,Tutorial_median_reading_time,Tutorial_std_reading_time,Tutorial_intensity_reading_time,Tutorial_total_count_hover,...,Memory_not_dominant_hand_z_range,Memory_dominant_hand_trigger_press_count,Memory_not_dominant_hand_trigger_press_count,Memory_dominant_hand_trigger_pressure_mean,Memory_not_dominant_hand_trigger_pressure_mean,Memory_dominant_hand_grip_count,Memory_not_dominant_hand_grip_count,Memory_dominant_hand_grip_mean,Gender_Male,dominant_hand_Right
0,73.0,1,28.0,45.03,15.01,21.00,17.23,7.36,0.92,32,...,0.50,35,0,0.01,0.0,1275,0,0.40,0,1
1,59.0,3,27.0,171.99,57.33,91.56,69.81,41.89,0.98,322,...,1.15,63,0,0.02,0.0,222,0,0.09,0,1
2,82.0,4,26.0,338.75,112.92,155.14,147.97,67.02,0.99,688,...,0.61,12,0,0.00,0.0,277,0,0.09,1,1
3,75.0,3,27.0,114.78,38.26,68.19,43.00,32.56,0.97,112,...,0.92,47,1,0.03,0.0,157,0,0.10,1,0
4,62.0,4,27.0,152.90,50.97,88.47,42.24,33.99,0.97,61,...,0.48,28,0,0.02,0.0,125,0,0.08,1,1


In [ ]:
def create_correlation_series(df: pd.DataFrame, target_col: str) -> pd.Series:
    """
    Create correlation series between columns and target.
    Args : 
        df(pd.DataFrame) : dataframe.
        target_col(str): target column.
    Returns :   
        series of correaltion with target and all columns.
    """
    return df.corr()[target_col].sort_values(ascending=False)

def create_nan_correlation(correlation_series: pd.Series) -> list:
    """
    Create list of Nan correlation columns with target.
    Args : 
        correlation_series(pd.Series) : series of correlation between columns and target.
        threshold(float): less than this, remove it.
    Returns :   
        List
    """
    return correlation_series[correlation_series.isna()].index.tolist()

def find_small_corr_cols_with_threshold(correlation_series: pd.Series, threshold: float) -> list:
    """
    Find small correlations by threshold.
    Args : 
        correlation_series(pd.Series) : series of correlation between columns and target.
        threshold(float): less than this, remove it.
    Returns :   
        List
    """
    return correlation_series[abs(correlation_series) < threshold].index.tolist()

def create_correlation_matrix_abs(df: pd.DataFrame, selected_cols: list)-> pd.DataFrame:
    """
    create correlation matrix with abs values from columns with
    high correlation with target.
    Args : 
        df(pd.DataFrame) : df
        selected_cols(list) : list of columns.
    Returns :   
        feature matrix (dataframe)
    """
    return df[selected_cols].corr().abs()

def find_redundant_features(feature_corr: pd.DataFrame)-> list:
    """
    Find the columns with high correlation, return them as a list.
    Args : 
        feature_corr(pd.DataFrame): Feature correlation matirx.
    Returns :   
        Lists of columns with high correlations (they are redundant).
    """
    upper = feature_corr.where(np.triu(np.ones(feature_corr.shape), k=1).astype(bool))
    return [column for column in upper.columns if any(upper[column] > 0.9)]
    
def treat_useless_correlations(df: pd.DataFrame, useless_corr: list)-> pd.DataFrame:
    """
    Remove all small or nan correlation and redudant columns.
    Args : 
        df(pd.DataFrame): Cleaned dataframe without Nan values.
        useless_corr (list): list of useless columns.
    Returns :   
        df with no useless columns.
    """
    df = df.drop(columns= useless_corr)
    return df

def check_correlation(df: pd.DataFrame):
    """
    Check correlations between target-columns, and column_column.
    Args : 
        df (pd.DataFrame): output df of check_constant_column func.
    Returns :   
        cleaned_df
    """
    all_columns     = df_cleaning.columns.tolist()
    useless_columns = []

    print(f'Number of columns before removing {len(df.columns)}')

    correlation_series     = create_correlation_series(df, "MoCA_Score")
    nan_correlation_series = create_nan_correlation(correlation_series)

    if len(nan_correlation_series) > 0 :
        print(f'Number of Nan correlation {len(nan_correlation_series)}')
        useless_columns.extend(nan_correlation_series)
    else:
        print('No Nan correlation.')

    small_corr = find_small_corr_cols_with_threshold(correlation_series, 0.2)

    if len(small_corr) > 0 :
        print(f'Number of small correlation {len(small_corr)}')
        useless_columns.extend(small_corr)
    else:
        print('No small correlation')
    
    useful_columns = [col for col in all_columns if col not in useless_columns]

    feature_corr = create_correlation_matrix_abs(df, useful_columns)
    redundant_features = find_redundant_features(feature_corr)

    if len(redundant_features) > 0 :
        print(f'Number of redundant features {len(redundant_features)}')
        useless_columns.extend(redundant_features)
    else:
        print('No redundant features')
    
    if len(useless_columns)> 0 :
        print(f'Number of total useless columns {len(useless_columns)}, Removing them...')
        df =  treat_useless_correlations(df, useless_columns)
        print('All useless correlations are removed successfully.')
    else :
        print('No useless correlation is found!')
    
    print(f'Number of columns after removing {len(df.columns)}')

    return df

In [33]:
df_cleaning = check_correlation(df_cleaning)

Number of columns before removing 347
No Nan correlation.
Number of small correlation 162
Number of redundant features 65
Number of total useless columns 227, Removing them...
All useless correlations are removed successfully.
Number of columns after removing 120


In [29]:
all_columns = df_cleaning.columns.tolist()
useless_col = check_correlation(df_cleaning)
col_keep = [col for col in all_columns if col not in useless_col]
len(col_keep)

No Nan correlation.
Number of small correlation 162


185

In [25]:
check_correlation(df_cleaning)

No Nan correlation.
Number of small correlation 162


['Visuospatial_hover_vs_active_interaction_ratio',
 'ObjectRecognition_Yaw_std',
 'Tutorial_max_duration_gaze',
 'ObjectRecognition_std_head_speed_in_orientation',
 'Memory_median_duration_gaze',
 'Visuospatial_median_duration_hover',
 'ObjectRecognition_max_duration_gaze',
 'Tutorial_std_duration_gaze',
 'Tutorial_dominant_hand_mean_speed',
 'Memory_not_dominant_hand_max_speed',
 'ObjectRecognition_HMD_Z_std',
 'ObjectRecognition_interaction_fraction',
 'ObjectRecognition_intensity_hover',
 'Tutorial_std_duration_press',
 'Memory_dominant_hand_y_range',
 'ObjectRecognition_mean_choose_wrong_obj',
 'Memory_hover_vs_active_interaction_ratio',
 'ObjectRecognition_head_total_orientation',
 'ObjectRecognition_time_before_first_press',
 'Visuospatial_head_total_orientation',
 'Tutorial_max_duration_press',
 'Memory_dominant_hand_x_range',
 'Tutorial_mean_duration_gaze',
 'Tutorial_intensity_press',
 'Tutorial_hover_vs_active_interaction_ratio',
 'Memory_max_head_speed_in_orientation',
 'Obj

In [9]:
# correlation
correlation_series = df_cleaning.corr()["MoCA_Score"].sort_values(ascending=False)
correlation_series

MoCA_Score                                    1.000000
Visuospatial_Roll_range                       0.868104
dominant_hand_Right                           0.637675
ObjectRecognition_mean_reading_time           0.571935
Visuospatial_std_head_speed_in_orientation    0.551825
                                                ...   
Memory_dominant_hand_trigger_press_count     -0.851592
Visuospatial_mean_duration_gaze              -0.861446
Visuospatial_total_duration_gaze             -0.885136
Tutorial_not_dominant_hand_grip_count        -0.888653
Tutorial_not_dominant_hand_grip_mean         -0.891330
Name: MoCA_Score, Length: 347, dtype: float64

In [10]:
nan_corr_cols = correlation_series[correlation_series.isna()].index.tolist()
nan_corr_cols

[]

In [11]:
if len(nan_corr_cols) > 0:
    df_cleaning.drop(columns = nan_corr_cols)

selected_cols = correlation_series[abs(correlation_series) >= 0.2].index.tolist()

In [12]:
len(selected_cols)

185

In [17]:
feature_corr = df_cleaning[selected_cols].corr().abs()
feature_corr

,MoCA_Score,Visuospatial_Roll_range,dominant_hand_Right,ObjectRecognition_mean_reading_time,Visuospatial_std_head_speed_in_orientation,ObjectRecognition_total_reading_time,Visuospatial_clicks_per_second,Visuospatial_std_head_speed_in_distance,Help_Rating_out_of_5,Visuospatial_mean_head_speed_in_orientation,...,Visuospatial_std_duration_gaze,Visuospatial_std_reading_time,ObjectRecognition_dominant_hand_trigger_press_count,Visuospatial_dominant_hand_trigger_press_count,Memory_not_dominant_hand_grip_count,Memory_dominant_hand_trigger_press_count,Visuospatial_mean_duration_gaze,Visuospatial_total_duration_gaze,Tutorial_not_dominant_hand_grip_count,Tutorial_not_dominant_hand_grip_mean
MoCA_Score,1.000000,0.868104,0.637675,0.571935,0.551825,0.529368,0.497503,0.490319,0.480136,0.460140,...,0.801957,0.831381,0.834499,0.844594,0.847476,0.851592,0.861446,0.885136,0.888653,0.891330
Visuospatial_Roll_range,0.868104,1.000000,0.670956,0.601328,0.629006,0.433467,0.420152,0.393974,0.526578,0.534466,...,0.689643,0.849992,0.827858,0.823553,0.858026,0.808530,0.811512,0.897853,0.931037,0.932280
dominant_hand_Right,0.637675,0.670956,1.000000,0.444444,0.346144,0.497067,0.228436,0.410389,0.390567,0.296156,...,0.496100,0.615783,0.619460,0.580131,0.631442,0.586761,0.732155,0.643286,0.686896,0.688247
ObjectRecognition_mean_reading_time,0.571935,0.601328,0.444444,1.000000,0.033125,0.688247,0.337216,0.330959,0.390567,0.088413,...,0.486420,0.645603,0.593898,0.591191,0.631442,0.569127,0.527104,0.607621,0.686896,0.688247
Visuospatial_std_head_speed_in_orientation,0.551825,0.629006,0.346144,0.033125,1.000000,0.004918,0.502139,0.466102,0.251184,0.984597,...,0.663378,0.464453,0.512096,0.512471,0.540371,0.463284,0.697379,0.555747,0.503598,0.501386
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Memory_dominant_hand_trigger_press_count,0.851592,0.808530,0.586761,0.569127,0.463284,0.544968,0.680468,0.621937,0.631387,0.390965,...,0.723511,0.844963,0.868659,0.977103,0.826638,1.000000,0.754637,0.930061,0.891365,0.892769
Visuospatial_mean_duration_gaze,0.861446,0.811512,0.732155,0.527104,0.697379,0.429190,0.580163,0.623995,0.450839,0.625147,...,0.877784,0.745894,0.784991,0.772982,0.836638,0.754637,1.000000,0.854336,0.828266,0.827664
Visuospatial_total_duration_gaze,0.885136,0.897853,0.643286,0.607621,0.555747,0.468153,0.676642,0.628688,0.517667,0.464726,...,0.795320,0.861419,0.864608,0.928221,0.896281,0.930061,0.854336,1.000000,0.943241,0.943190
Tutorial_not_dominant_hand_grip_count,0.888653,0.931037,0.686896,0.686896,0.503598,0.471261,0.531527,0.547757,0.564860,0.407290,...,0.742869,0.934650,0.912608,0.915003,0.942256,0.891365,0.828266,0.943241,1.000000,0.999862


In [14]:
import numpy as np

upper = feature_corr.where(
    np.triu(np.ones(feature_corr.shape), k=1).astype(bool)
)
upper

,MoCA_Score,Visuospatial_Roll_range,dominant_hand_Right,ObjectRecognition_mean_reading_time,Visuospatial_std_head_speed_in_orientation,ObjectRecognition_total_reading_time,Visuospatial_clicks_per_second,Visuospatial_std_head_speed_in_distance,Help_Rating_out_of_5,Visuospatial_mean_head_speed_in_orientation,...,Visuospatial_std_duration_gaze,Visuospatial_std_reading_time,ObjectRecognition_dominant_hand_trigger_press_count,Visuospatial_dominant_hand_trigger_press_count,Memory_not_dominant_hand_grip_count,Memory_dominant_hand_trigger_press_count,Visuospatial_mean_duration_gaze,Visuospatial_total_duration_gaze,Tutorial_not_dominant_hand_grip_count,Tutorial_not_dominant_hand_grip_mean
MoCA_Score,NaN,0.868104,0.637675,0.571935,0.551825,0.529368,0.497503,0.490319,0.480136,0.460140,...,0.801957,0.831381,0.834499,0.844594,0.847476,0.851592,0.861446,0.885136,0.888653,0.891330
Visuospatial_Roll_range,NaN,NaN,0.670956,0.601328,0.629006,0.433467,0.420152,0.393974,0.526578,0.534466,...,0.689643,0.849992,0.827858,0.823553,0.858026,0.808530,0.811512,0.897853,0.931037,0.932280
dominant_hand_Right,NaN,NaN,NaN,0.444444,0.346144,0.497067,0.228436,0.410389,0.390567,0.296156,...,0.496100,0.615783,0.619460,0.580131,0.631442,0.586761,0.732155,0.643286,0.686896,0.688247
ObjectRecognition_mean_reading_time,NaN,NaN,NaN,NaN,0.033125,0.688247,0.337216,0.330959,0.390567,0.088413,...,0.486420,0.645603,0.593898,0.591191,0.631442,0.569127,0.527104,0.607621,0.686896,0.688247
Visuospatial_std_head_speed_in_orientation,NaN,NaN,NaN,NaN,NaN,0.004918,0.502139,0.466102,0.251184,0.984597,...,0.663378,0.464453,0.512096,0.512471,0.540371,0.463284,0.697379,0.555747,0.503598,0.501386
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Memory_dominant_hand_trigger_press_count,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.754637,0.930061,0.891365,0.892769
Visuospatial_mean_duration_gaze,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.854336,0.828266,0.827664
Visuospatial_total_duration_gaze,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.943241,0.943190
Tutorial_not_dominant_hand_grip_count,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.999862


In [15]:
to_drop = [
    column
    for column in upper.columns
    if any(upper[column] > 0.9)
]

print(len(to_drop))

61


## 2.2 Standardize (z-score)

## 2.3 Exploratory Data Analysis

correlation matrix

variance filtering

redundancy detection


## 2.4 Reduce dimensionality (PCA)

## 2.5 Validate

correlation preservation

distribution matching

model stability